# Route B Pipeline 1: Histogram Gradient Boosting

Predicts `career_games_started` from combine measurements, validates locally, and saves held-out predictions.

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Config (edit these)

In [10]:
COMBINE_PATH = "data/combine_results.csv"
CAREER_PATH = "data/player_career_successes.csv"
TARGET = "career_games_started"
RANDOM_STATE = 307

FEATURES = ["draft_year", "combine_position", "combine_height", "combine_weight",
            "hand_size", "arm_length", "wing_span", "forty", "bench_reps",
            "vertical", "broad_jump", "three_cone", "short_shuttle", "ten_yd_split"]

## 2. Preparation: one row per player (combine + career outcome)

In [11]:
combine = pd.read_csv(COMBINE_PATH)
career = pd.read_csv(CAREER_PATH)

missing_cols = [c for c in FEATURES if c not in combine.columns]
if missing_cols:
    raise KeyError(f"Feature columns not found in combine data: {missing_cols}")
features = FEATURES

train = combine[["nfl_id"] + features].merge(
    career[["nfl_id", TARGET]], on="nfl_id", how="inner", validate="one_to_one")
train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")
train = train.dropna(subset=[TARGET])
X, y = train[features], train[TARGET]

numeric = X.select_dtypes(include="number").columns.tolist()
categorical = X.select_dtypes(exclude="number").columns.tolist()
print("Training table:", train.shape)
print("Numeric:", numeric)
print("Categorical:", categorical)

Training table: (510, 16)
Numeric: ['draft_year', 'combine_height', 'combine_weight', 'hand_size', 'arm_length', 'wing_span', 'forty', 'bench_reps', 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'ten_yd_split']
Categorical: ['combine_position']


## 3. Pipeline definition (preprocessing lives inside the pipeline)

In [12]:
def make_pipeline():
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])
    return Pipeline([
        ("pre", pre),
        ("model", HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.05, l2_regularization=1.0,
            random_state=RANDOM_STATE)),
    ])

## 4. Local validation: 75/25 holdout and 5-fold CV

In [14]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)
base = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
pipe = make_pipeline().fit(X_tr, y_tr)


def metrics(name, model):
    p = model.predict(X_val)
    return {"Model": name,
            "MAE": mean_absolute_error(y_val, p),
            "RMSE": np.sqrt(mean_squared_error(y_val, p)),
            "R2": r2_score(y_val, p)}


val = pd.DataFrame([metrics("Mean baseline", base),
                    metrics("HistGradientBoosting", pipe)])
cv_mae = -cross_val_score(make_pipeline(), X, y, scoring="neg_mean_absolute_error",
                          cv=KFold(5, shuffle=True, random_state=RANDOM_STATE))
val["CV5_MAE_mean"] = [np.nan, cv_mae.mean()]
val.to_csv("validation_metrics.csv", index=False)
val.round(3)

,Model,MAE,RMSE,R2,CV5_MAE_mean
0,Mean baseline,8.296,10.878,-0.000,NaN
1,HistGradientBoosting,8.965,11.631,-0.143,8.581


## 5. Inspect and save held-out predictions

In [17]:
holdout = pd.DataFrame({
    "nfl_id": train.loc[X_val.index, "nfl_id"],
    "actual_games_started": y_val,
    "predicted_games_started": np.clip(pipe.predict(X_val), 0, None),  # starts cannot be negative
})
holdout["absolute_error"] = (holdout["actual_games_started"] - holdout["predicted_games_started"]).abs()
holdout.to_csv("holdout_predictions.csv", index=False)
print("Wrote holdout_predictions.csv", holdout.shape)
holdout.sort_values("absolute_error", ascending=False).head(15)

Wrote holdout_predictions.csv (128, 4)


,nfl_id,actual_games_started,predicted_games_started,absolute_error
155,55924,50,12.385195,37.614805
90,55960,30,0.000000,30.000000
44,55958,29,0.000000,29.000000
223,57155,26,0.000000,26.000000
157,56183,0,25.380958,25.380958
305,57207,34,9.025896,24.974104
118,55910,37,14.753230,22.246770
292,57128,30,8.163185,21.836815
216,57228,26,4.873873,21.126127
83,55919,36,15.189061,20.810939
